In [ ]:
# Configuracao do ambiente (celula de setup do notebook)
import sys, os, warnings
sys.path.insert(0, os.path.abspath(os.path.join('..', '..', 'src')))
warnings.filterwarnings('ignore')
%matplotlib inline

# Capítulo 10 — A SABEDORIA DAS ÁRVORES: DECISION TREES E ENSEMBLES

> "Uma única árvore pode ser enganada pelo vento, vendo apenas a direção de uma rajada. Mas a floresta, sentindo o vento de mil direções, compreende a tempestade inteira."
> — Provérbio dos Modeladores de Ensembles

O SVM me deu uma fronteira geométrica elegante, mas abstrata. Como eu explicaria uma previsão dele ao chefe da oncologia? "O modelo mapeou os dados para um espaço de alta dimensão e encontrou um hiperplano ótimo" — ele me olharia com desconfiança, e com razão.

Eu precisava de um algoritmo que pensasse como um médico: fazendo perguntas. "O raio é maior que 15? A textura é irregular? Há pontos côncavos?" Cada resposta leva à próxima, até o diagnóstico. Isso é uma **Árvore de Decisão** — lindamente interpretável. Mas uma única árvore, como um único especialista, tem pontos cegos e tende a *superajustar*. A verdadeira força vem de consultar não uma árvore, mas uma floresta inteira.

## Árvores de Decisão e o Poder dos Ensembles

Uma **Árvore de Decisão** divide os dados em subconjuntos cada vez mais "puros" por meio de perguntas sobre as *features* (ex.: `worst radius <= 16.8`). Sua grande vantagem é a **interpretabilidade** (modelo de "caixa-branca"); sua fraqueza, a tendência ao **overfitting** — pode crescer até memorizar o ruído do treino.

Para combater isso, usamos **ensembles**, que combinam muitos modelos simples. Duas filosofias:

**Bagging (ex.: Random Forest)** — treina centenas de árvores *independentes*, cada uma numa amostra aleatória (com reposição) e vendo só um subconjunto de *features*. A previsão final é o voto da maioria. Uma democracia de especialistas.

**Boosting (ex.: Gradient Boosting)** — treina árvores em *sequência*, cada uma corrigindo os erros da anterior. Uma equipe onde cada novo membro aprende com as falhas dos anteriores.

Árvores **não são sensíveis à escala**, então dispensamos o `StandardScaler`.

#### Código 10.1: Da Árvore à Floresta


In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import cross_validate
from oncoclassify_utils import X_train, y_train, recall_maligno

metricas = {"acuracia": "accuracy", "recall_Maligno": recall_maligno}
modelos = {
    "Árvore de Decisão": DecisionTreeClassifier(random_state=42),
    "Random Forest":     RandomForestClassifier(n_estimators=100, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, random_state=42),
}
for nome, modelo in modelos.items():
    r = cross_validate(modelo, X_train, y_train, cv=5, scoring=metricas)
    print(f"{nome:20s} acuracia {r['test_acuracia'].mean():.4f} | "
          f"recall Maligno {r['test_recall_Maligno'].mean():.4f}")

Árvore de Decisão    acuracia 0.9479 | recall Maligno 0.9265
Random Forest        acuracia 0.9583 | recall Maligno 0.9319
Gradient Boosting    acuracia 0.9563 | recall Maligno 0.9265


A história é clara. A **árvore única** atinge 94,79% — decente, mas o menor do trio, penalizada pela sua tendência ao *overfitting*. Os **ensembles** sobem: Random Forest a **95,83%** e Gradient Boosting a **95,63%**, ambos com recall maligno em torno de 93%. Combinar muitas árvores diversas cancela o ruído das individuais.

Vale notar, porém, que neste dataset os ensembles de árvores **não** superaram o SVM-RBF (97,5%). Nem sempre o modelo mais sofisticado vence — uma lição que consolidaremos na comparação final (Capítulo 30).

> **📌 CHECKLIST DO CAPÍTULO 10**
>
> ✅ Entende como uma Árvore de Decisão divide os dados por perguntas.
>
> ✅ Sabe sua vantagem (interpretabilidade) e desvantagem (overfitting).
>
> ✅ Compreende **Bagging** (Random Forest) vs **Boosting** (Gradient Boosting).
>
> ✅ Executou o Código 10.1 e viu o salto da árvore única (**94,8%**) para os ensembles (~**95,8%**).
>
> **⚠️ CRÍTICO:** Ensembles de árvores estão entre os algoritmos mais usados para dados tabulares. Mas "mais poderoso" não é garantia de "melhor aqui": o SVM-RBF ainda lidera.

Eu tinha agora uma floresta à disposição. A ideia de que centenas de modelos simples, combinados, superam um único especialista era uma lição poderosa — não só em *machine learning*, mas na vida. A robustez vinha da diversidade.

Eu havia explorado a probabilidade (Bayes), a geometria (SVM) e a multidão (árvores). Faltava um algoritmo talvez o mais intuitivo de todos, baseado numa ideia profundamente humana: "diga-me com quem andas e eu te direi quem és". Era hora de explorar a vizinhança.
